# Score independent cohort against **experimental PDB backbones** - MPNN family

Scores the 876-chain independent cohort
(`design/outputs/independent_cohort/cohort_pdb_scoring_inputs.csv`) using the
**real PDB chains** as the backbone $\mathbf{X}$, for the autoregressive
conditional log-likelihood (Eq. AR):

$$\ell_\text{AR}(\mathbf{s}\mid\mathbf{X})=\frac1L\sum_{i=1}^L \log p\big(s_{\pi(i)}\mid s_{\pi(<i)},\mathbf{X}\big)$$

**Models** (all via the official `protein_mpnn_run.py --score_only 1`; we store
`-global_score = ELL_AR`, per-residue mean, higher-is-better):

| subdir | checkpoint | flag |
|---|---|---|
| `proteinmpnn` | `v_48_020` | - |
| `solublempnn` | `v_48_020` | `--use_soluble_model` |
| `v_48_002_base` | `v_48_002` | - (matched FT base) |
| `AlkalineMPNN` | FT `.pt` | `--path_to_model_weights` |
| `AcidophileMPNN` | FT `.pt` | `--path_to_model_weights` |

**Backbone source:** unlike the AlphaFold scorer, this notebook reads each chain
from the pre-extracted single-chain PDBs in the cohort bundle (chain `A` in every
file). No AlphaFold/RCSB download. The CLI reads the native sequence directly
from the PDB ATOM records (= the cohort `resolved` sequence), so AR teacher-forcing
and the sign convention carry over unchanged.

**You need to upload two zips:**
1. `finetune/colab/ft_mpnn_weights.zip`  -> `AlkalineMPNN.pt`, `AcidophileMPNN.pt`
2. `cohort_pdb_scoring_bundle.zip`  (from `design/outputs/independent_cohort/`) -> 876 chain PDBs + the input CSV


In [ ]:
#@title Drive mount + output dir
import os
try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    DRIVE_RESULTS_DIR = '/content/drive/MyDrive/decoding_bias_results/cohort_pdb'
except Exception as _e:
    print('Not in Colab? ', _e)
    DRIVE_RESULTS_DIR = '/content/cohort_pdb_results'
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print('Results ->', DRIVE_RESULTS_DIR)

In [ ]:
#@title Resume-from-checkpoint helpers
import time as _t, glob as _g
import pandas as pd
SESSION_ID = _t.strftime("%Y%m%d_%H%M%S")

def get_completed_entries(model_dir):
    if not os.path.isdir(model_dir):
        return set()
    done = set()
    for fn in _g.glob(os.path.join(model_dir, "*.csv")):
        try:
            _df = pd.read_csv(fn)
            if "Entry" in _df.columns:
                done.update(_df["Entry"].dropna().unique())
        except Exception as _e:
            print(f"  could not read {fn}: {_e}")
    return done

def csv_filter_resume(input_csv, model_dir):
    done = get_completed_entries(model_dir)
    if not done:
        print(f"  Resume: no prior results in {model_dir}; running full CSV")
        return input_csv
    df = pd.read_csv(input_csv)
    before = len(df)
    df = df[~df["Entry"].isin(done)].reset_index(drop=True)
    print(f"  Resume: {len(done)} done in {model_dir}; {len(df)}/{before} remaining (session {SESSION_ID})")
    if len(df) == 0:
        print("  All entries already scored - nothing to do.")
        return None
    out = input_csv.replace(".csv", f"_remaining_{SESSION_ID}.csv")
    df.to_csv(out, index=False)
    return out

In [ ]:
#@title Clone dauparas/ProteinMPNN (official CLI; soluble weights bundled)
import subprocess, sys
PMPNN_REPO_DIR = '/content/ProteinMPNN_dauparas'
if not os.path.isdir(PMPNN_REPO_DIR):
    print('Cloning dauparas/ProteinMPNN...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/dauparas/ProteinMPNN.git',
                    PMPNN_REPO_DIR], check=True)
PMPNN_RUN_PY   = os.path.join(PMPNN_REPO_DIR, 'protein_mpnn_run.py')
PMPNN_PARSE_PY = os.path.join(PMPNN_REPO_DIR, 'helper_scripts', 'parse_multiple_chains.py')
assert os.path.exists(PMPNN_RUN_PY) and os.path.exists(PMPNN_PARSE_PY)
print('CLI ready:', PMPNN_RUN_PY)

In [ ]:
#@title Upload fine-tuned weights  (finetune/colab/ft_mpnn_weights.zip)
import zipfile, torch
FT_WEIGHTS_DIR = "/content/ft_weights"
if not (os.path.isdir(FT_WEIGHTS_DIR) and any(f.endswith(".pt") for f in os.listdir(FT_WEIGHTS_DIR))):
    print("Upload finetune/colab/ft_mpnn_weights.zip:")
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall("/content")
for f in os.listdir(FT_WEIGHTS_DIR):
    if f.endswith(".pt"):
        fp = os.path.join(FT_WEIGHTS_DIR, f); ck = torch.load(fp, map_location="cpu")
        if "noise_level" not in ck:
            ck["noise_level"] = 0.2; torch.save(ck, fp); print("  added noise_level ->", f)
print("FT weights ready:", sorted(os.listdir(FT_WEIGHTS_DIR)))

In [ ]:
#@title Upload cohort bundle  (cohort_pdb_scoring_bundle.zip) and build Entry->PDB map
import zipfile, glob
COHORT_DIR = "/content/cohort"
os.makedirs(COHORT_DIR, exist_ok=True)
if not glob.glob(os.path.join(COHORT_DIR, "cohort_chain_structs", "*.pdb")):
    print("Upload cohort_pdb_scoring_bundle.zip (from design/outputs/independent_cohort/):")
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall(COHORT_DIR)

COHORT_CSV   = os.path.join(COHORT_DIR, "cohort_pdb_scoring_inputs.csv")
STRUCT_DIR   = os.path.join(COHORT_DIR, "cohort_chain_structs")
_cdf = pd.read_csv(COHORT_CSV)

# Map each Entry to its LOCAL chain PDB (basename of the original chain_pdb_path).
ENTRY2PDB = {}
for _, r in _cdf.iterrows():
    local = os.path.join(STRUCT_DIR, os.path.basename(r["chain_pdb_path"]))
    ENTRY2PDB[r["Entry"]] = local
missing = [e for e, p in ENTRY2PDB.items() if not os.path.exists(p)]
print(f"cohort rows: {len(_cdf)} | PDBs mapped: {len(ENTRY2PDB)} | missing: {len(missing)}")
assert not missing, f"missing PDBs: {missing[:5]}"
print("every chain uses id 'A' in the extracted files -> chain='A'")

In [ ]:
#@title Processor classes (official CLI; backbone = local cohort chain PDB)
import tempfile, shutil, logging
import numpy as np
from tqdm import tqdm
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

class SolubleMPNNProcessor:
    """dauparas CLI --use_soluble_model --score_only 1.
    sequence_score = -global_score (per-residue log-lik, higher=better)."""
    def __init__(self, model_name='v_48_020', batch_size=1):
        self.model_name = model_name; self.batch_size = batch_size
        self.use_soluble = True; self.weights_dir = ""

    # backbone = local cohort chain PDB (no download)
    def get_pdb(self, entry, output_dir=None):
        p = ENTRY2PDB.get(entry)
        return (p, True) if p and os.path.exists(p) else (None, False)

    def _score_chunk(self, accession_to_pdb):
        if not accession_to_pdb:
            return []
        workdir = tempfile.mkdtemp(prefix="mpnn_")
        pdb_in = os.path.join(workdir, "pdbs"); out_dir = os.path.join(workdir, "out")
        os.makedirs(pdb_in, exist_ok=True); os.makedirs(out_dir, exist_ok=True)
        jsonl = os.path.join(workdir, "parsed.jsonl"); staged = {}
        for entry, src in accession_to_pdb.items():
            shutil.copyfile(src, os.path.join(pdb_in, f"{entry}.pdb")); staged[entry] = entry
        subprocess.run([sys.executable, PMPNN_PARSE_PY, "--input_path", pdb_in,
                        "--output_path", jsonl], check=True)
        cmd = [sys.executable, PMPNN_RUN_PY, "--jsonl_path", jsonl, "--out_folder", out_dir,
               "--score_only", "1", "--model_name", self.model_name,
               "--batch_size", str(self.batch_size), "--suppress_print", "1"]
        if self.use_soluble:
            cmd += ["--use_soluble_model"]
        if self.weights_dir:
            cmd += ["--path_to_model_weights", self.weights_dir]
        subprocess.run(cmd, check=True)
        results = []
        for tag, entry in staged.items():
            g = glob.glob(os.path.join(out_dir, "score_only", f"{tag}*.npz"))
            if not g:
                logging.warning(f"No score npz for {entry}"); continue
            d = np.load(g[0])
            results.append({"Entry": entry,
                            "sequence_score": -float(np.mean(d["global_score"])),
                            "entropy": np.nan,
                            "sequence_length": int(d["S"].shape[-1]) if "S" in d else None,
                            "mean_confidence": np.nan})
        shutil.rmtree(workdir, ignore_errors=True)
        return results

    def process_proteins_in_chunks(self, csv_file, chunk_size=500, chain="A", subdir=None):
        subdir = subdir or self.model_name.lower()
        df_in = pd.read_csv(csv_file); total = len(df_in); all_results = []
        logging.info(f"{subdir}: scoring {total} proteins in chunks of {chunk_size}")
        for start in range(0, total, chunk_size):
            end = min(start + chunk_size, total); chunk = df_in.iloc[start:end]; acc2pdb = {}
            for _, row in tqdm(chunk.iterrows(), total=len(chunk), desc=f"{subdir} [{start}:{end}]"):
                pth, ok = self.get_pdb(row["Entry"])
                if ok: acc2pdb[row["Entry"]] = pth
                else:  logging.warning(f"no PDB for {row['Entry']}")
            res = self._score_chunk(acc2pdb); all_results.extend(res)
            cp = os.path.join(DRIVE_RESULTS_DIR, subdir, f"{subdir}_results_{SESSION_ID}_chunk_{start}-{end-1}.csv")
            os.makedirs(os.path.dirname(cp), exist_ok=True); pd.DataFrame(res).to_csv(cp, index=False)
            logging.info(f"saved {len(res)} -> {cp}")
        final = pd.DataFrame(all_results)
        fp = os.path.join(DRIVE_RESULTS_DIR, subdir, f"{subdir}_results_all_{SESSION_ID}.csv")
        os.makedirs(os.path.dirname(fp), exist_ok=True); final.to_csv(fp, index=False)
        logging.info(f"final -> {fp}"); return final

class FinetunedMPNNProcessor(SolubleMPNNProcessor):
    """vanilla base (weights_dir='') or fine-tuned (weights_dir=FT dir). No soluble flag."""
    def __init__(self, model_name, weights_dir="", batch_size=1):
        super().__init__(model_name=model_name, batch_size=batch_size)
        self.use_soluble = False; self.weights_dir = weights_dir

In [ ]:
#@title RUN - all five MPNN-family models over the cohort PDBs
# (proteinmpnn / base / FT via FinetunedMPNNProcessor; solublempnn via SolubleMPNNProcessor)
CHUNK = 500

JOBS = [
    # subdir,           processor,                kwargs
    ("proteinmpnn",     FinetunedMPNNProcessor,   dict(model_name="v_48_020", weights_dir="")),
    ("solublempnn",     SolubleMPNNProcessor,     dict(model_name="v_48_020")),
    ("v_48_002_base",   FinetunedMPNNProcessor,   dict(model_name="v_48_002", weights_dir="")),
    ("AlkalineMPNN",    FinetunedMPNNProcessor,   dict(model_name="AlkalineMPNN",   weights_dir=FT_WEIGHTS_DIR)),
    ("AcidophileMPNN",  FinetunedMPNNProcessor,   dict(model_name="AcidophileMPNN", weights_dir=FT_WEIGHTS_DIR)),
]

for subdir, Proc, kw in JOBS:
    print(f"\n===== {subdir} =====")
    csvp = csv_filter_resume(COHORT_CSV, os.path.join(DRIVE_RESULTS_DIR, subdir))
    if csvp is None:
        print("  already complete (resume)"); continue
    Proc(**kw).process_proteins_in_chunks(csvp, chunk_size=CHUNK, subdir=subdir)

In [ ]:
#@title Merge all models into one wide table (ELL_AR, higher = better)
import glob
def _load(subdir):
    f = os.path.join(DRIVE_RESULTS_DIR, subdir, f"{subdir}_results_all_{SESSION_ID}.csv")
    if os.path.exists(f):
        return pd.read_csv(f)
    ch = glob.glob(os.path.join(DRIVE_RESULTS_DIR, subdir, f"{subdir}_results_*chunk*.csv"))
    return pd.concat([pd.read_csv(c) for c in ch], ignore_index=True) if ch else pd.DataFrame()

SUBDIRS = ["proteinmpnn", "solublempnn", "v_48_002_base", "AlkalineMPNN", "AcidophileMPNN"]
wide = _cdf[["Entry", "pdb_id", "chain", "domain", "protein_family", "broad_function"]].copy()
for s in SUBDIRS:
    d = _load(s)
    if d.empty:
        print(f"  no results for {s}"); continue
    wide = wide.merge(d[["Entry", "sequence_score"]].rename(columns={"sequence_score": s}),
                      on="Entry", how="left")
    print(f"  {s}: {d['Entry'].nunique()} scored")

out = os.path.join(DRIVE_RESULTS_DIR, f"cohort_pdb_mpnn_scores_{SESSION_ID}.csv")
wide.to_csv(out, index=False)
print("\nwide table ->", out)
wide.head()